In [ ]:
import os
os.environ["TRANSFORMERS_NO_VISION"] = "1"

!pip install --upgrade torch==2.2.0 transformers==4.40.0 datasets accelerate bitsandbytes nltk tqdm scikit-learn matplotlib seaborn pandas
!pip install sentencepiece

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, BertTokenizer, BertModel, AutoModel, pipeline
from datasets import Dataset, DatasetDict, load_dataset
from huggingface_hub import login
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict, Counter
from tqdm import tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statistics
import random
import json
import re


HF_L = "XXX"
login(token=HF_L)

In [ ]:
model = 'qwen-7b'

if model == 'qwen-7b':
    qwen_model_name = "Qwen/Qwen2.5-7B-Instruct"
    qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
    qwen_model = AutoModelForCausalLM.from_pretrained(
        qwen_model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    qwen_pipeline = pipeline(
        "text-generation",
        model=qwen_model,
        tokenizer=qwen_tokenizer,
        torch_dtype=torch.float16,
        device_map="auto"
    )

if model == "phi3-mini-3.8b":
    phi_model_name = "microsoft/Phi-3-mini-4k-instruct"
    phi_tokenizer = AutoTokenizer.from_pretrained(phi_model_name)
    phi_model = AutoModelForCausalLM.from_pretrained(
        phi_model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    phi_pipeline = pipeline(
        "text-generation",
        model=phi_model,
        tokenizer=phi_tokenizer,
        torch_dtype=torch.float16,
        device_map="auto"
    )

if model == "smollm2-1.7b":
    smol_model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    smol_tokenizer = AutoTokenizer.from_pretrained(smol_model_name)
    smol_model = AutoModelForCausalLM.from_pretrained(
        smol_model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    smol_pipeline = pipeline(
        "text-generation",
        model=smol_model,
        tokenizer=smol_tokenizer,
        torch_dtype=torch.float16,
        device_map="auto"
    )


In [ ]:
def request_qwen(prompt, max_tokens=500, temperature=1.0, top_k=50, top_p=0.9, seed=42):
    messages = [
        {"role": "user", "content": prompt}
    ]
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    qwen_model.to(device)
    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = qwen_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True
        )
    input_length = model_inputs["input_ids"].shape[1]
    new_tokens = generated_ids[0][input_length:]
    response = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

def request_phi3(prompt, max_tokens=500, temperature=1.0, top_k=50, top_p=0.9, seed=42):
    messages = [
        {"role": "user", "content": prompt}
    ]
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    phi_model.to(device)
    text = phi_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = phi_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = phi_model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True
        )
    input_length = model_inputs["input_ids"].shape[1]
    new_tokens = generated_ids[0][input_length:]
    response = phi_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

def request_smollm2(prompt, max_tokens=500, temperature=1.0, top_k=50, top_p=0.9, seed=42):
    messages = [
        {"role": "user", "content": prompt}
    ]
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    smol_model.to(device)
    text = smol_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = smol_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = smol_model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True
        )
    input_length = model_inputs["input_ids"].shape[1]
    new_tokens = generated_ids[0][input_length:]
    response = smol_tokenizer.decode(new_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
ANSWER_EXPANSION_PROMPT = '''You are given a question and a short answer.

Your task is to slightly expand the answer by adding a small amount of relevant detail from the question.
Keep the response concise. Add approximately 8–12 additional words.
Do NOT change the meaning of the original answer.
Do NOT make it long or explanatory.
Do NOT output the original question.
Do NOT add more details which are not in the answer or question.

Question: "{0}"
Short Answer: "{1}"

Expanded Answer:'''

In [ ]:
dataset = load_dataset("Ramitha/squad-min-length-200-expanded-results")
df = pd.DataFrame(dataset['rawcases'])
df.info()
# model == "EuroLLM-9b"
for index, row in tqdm(df.iterrows(), total=len(df), desc="Generating Answers"):
    question = row['question']
    answer = row['answerGenerated']
    prompt = ANSWER_EXPANSION_PROMPT.format(question, answer)
    if row['expansion_model']=='qwen-7b' and model=='qwen-7b':
        response = request_qwen(prompt, temperature=0.7)
        df.at[index, 'answerGenerated'] = response
    if row['expansion_model']=='phi3-mini-3.8b' and model=='phi3-mini-3.8b':
        response = request_phi3(prompt, temperature=0.7)
        df.at[index, 'answerGenerated'] = response
    if row['expansion_model']=='smollm2-1.7b' and model=='smollm2-1.7b':
        response = request_smollm2(prompt, temperature=0.7)
        df.at[index, 'answerGenerated'] = response

hf_dataset = DatasetDict({
    'rawcases': Dataset.from_pandas(df)
})
hf_dataset.push_to_hub("Ramitha/squad-min-length-200-expanded-results")